# Chấm LLM sinh truy vấn SPARQL — chạy trên Kaggle

Chạy đúng bộ chấm của dự án, trên đúng 333 câu mà các model seq2seq đã chấm. Chỉ khác máy.

### Trước khi chạy, bật ba thứ trong phần cài đặt notebook

1. **Accelerator:** `GPU T4 x2` hoặc `GPU P100`
2. **Internet:** bật — cần để tải mã nguồn và model
3. **Secrets:** thêm secret tên `HF_TOKEN`, dùng token của tài khoản **đã bấm đồng ý giấy phép Gemma**

### Vì sao chạy ở đây

Card 6 GB ở nhà không chứa nổi bảng nhúng của Gemma-4-E2B. T4 và P100 có 16 GB nên chạy
thoải mái, và còn đủ chỗ cho model lớn hơn.

**Mọi dòng LLM nên chạy trên CÙNG một máy** để so được với nhau — kể cả Qwen, dù nó chạy
được ở nhà. Bảng seq2seq thì giữ nguyên kết quả đo trên máy cá nhân, và nói rõ trong báo cáo
là hai bảng đo trên hai phần cứng.

## 1. Xem Kaggle cấp card gì

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit("Chưa bật GPU. Vào Settings -> Accelerator -> GPU rồi chạy lại.")

for index in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(index)
    total = torch.cuda.get_device_properties(index).total_memory / 2**30
    print(f"GPU {index}: {name} · {total:.1f} GB")

print("bfloat16 chạy được:", torch.cuda.is_bf16_supported())
print("torch:", torch.__version__)

## 2. Cài thư viện

`compressed-tensors` cần cho bản Gemma nén sẵn; `bitsandbytes` cần khi tự nén 4-bit.

In [ ]:
import subprocess
import sys

packages = [
    "transformers>=5.14.1",
    "accelerate>=1.14.0",
    "bitsandbytes>=0.49.2",
    "compressed-tensors",
    "rdflib>=7.6.0",
    "owlrl>=7.6.2",
    "sentencepiece>=0.2.2",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

import importlib
import transformers

importlib.reload(transformers)
print("transformers:", transformers.__version__)

## 3. Đăng nhập Hugging Face

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

login(UserSecretsClient().get_secret("HF_TOKEN"))
print("đã đăng nhập")

## 4. Lấy mã nguồn, ontology và dataset

Xoá rồi tải lại từ đầu, không cập nhật tại chỗ — tránh chuyện tưởng đã cập nhật mà thật ra
vẫn chạy bản cũ.

In [ ]:
import os
import pathlib
import shutil
import subprocess
import sys

REPO = "https://github.com/vpthinh19/ontology-chatbot.git"
BRANCH = "ontology-v2"
WORK = "/kaggle/working/ontology-chatbot"

shutil.rmtree(WORK, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, WORK], check=True)
os.chdir(WORK)
if WORK + "/src" not in sys.path:
    sys.path.insert(0, WORK + "/src")

print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
print("có bộ chấm LLM:", pathlib.Path(WORK, "src/ontchatbot/cli/benchmark_llm.py").is_file())

## 5. Kiểm dữ liệu

In [ ]:
from ontchatbot.settings import DATASET_DIR

for split in ("train", "val", "test"):
    path = DATASET_DIR / (split + ".jsonl")
    print(split.ljust(6), sum(1 for _ in path.open(encoding="utf-8")), "dòng")

## 6. Chấm từng model

Câu chấm lấy từ `val`, ví dụ nhắc kèm lấy từ `train` — không rò rỉ.

Chạy thử vài câu trước để đo tốc độ, rồi mới chạy cả tập. Ước lượng: nhân số giây mỗi câu
với 333.

In [ ]:
import subprocess
import sys


def benchmark(model, shots=12, limit=0, extra=()):
    command = [
        sys.executable, "-m", "ontchatbot.cli.benchmark_llm",
        "--model", model,
        "--shots", str(shots),
        "--allow-download",
    ]
    if limit:
        command += ["--limit", str(limit)]
    command += list(extra)
    environment = dict(os.environ, PYTHONPATH="src",
                       PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    subprocess.run(command, cwd=WORK, env=environment, check=True)


# Thử 8 câu để đo tốc độ trước khi chạy cả tập.
benchmark("Qwen/Qwen3.5-2B", limit=8)

### Chạy đủ 333 câu

In [ ]:
benchmark("Qwen/Qwen3.5-2B")

In [ ]:
# Gemma-4-E2B: 16 GB đủ chỗ, không cần nén và không cần đẩy sang RAM.
benchmark("google/gemma-4-E2B-it-qat-q4_0-unquantized", limit=8)

In [ ]:
benchmark("google/gemma-4-E2B-it-qat-q4_0-unquantized")

### Muốn thêm trục quy mô

16 GB còn chỗ cho model 7-9 tỉ tham số nếu nén 4-bit. Thêm một dòng như vậy trả lời được
câu "model to hơn đáng bao nhiêu điểm" — thứ card 6 GB ở nhà không trả lời được.

Đổi tên model rồi thêm `extra=("--load-4bit",)`.

In [ ]:
# benchmark("<tên model 7-9B>", limit=8, extra=("--load-4bit",))

## 7. Gom kết quả

Kaggle chỉ giữ lại những gì nằm trong `/kaggle/working`, nên chép kết quả ra đó trước khi
kết thúc phiên.

In [ ]:
import json
import shutil
from pathlib import Path

source = Path(WORK) / "artifacts" / "llm-benchmark"
destination = Path("/kaggle/working/ket-qua-llm")
destination.mkdir(parents=True, exist_ok=True)

rows = []
for path in sorted(source.glob("*.json")):
    shutil.copy2(path, destination / path.name)
    report = json.loads(path.read_text(encoding="utf-8"))
    overall, run = report["overall"], report.get("run", {})
    rows.append((run.get("model", path.stem), run.get("shots"), overall["count"],
                 overall["answer_exact_rate"], overall["system_answer_exact_rate"],
                 overall["safe_rejection_rate"], overall["parse_rate"],
                 run.get("seconds_per_question")))

header = f"{'model':44s} {'shot':>4s} {'câu':>4s} {'Đúng':>7s} {'HệThống':>8s} {'TừChối':>7s} {'SPARQL':>7s} {'giây':>6s}"
print(header)
print("-" * len(header))
for model, shots, count, exact, system, refusal, parse, speed in rows:
    print(f"{model[:44]:44s} {str(shots):>4s} {count:4d} {exact:7.1%} {system:8.1%} "
          f"{refusal:7.1%} {parse:7.1%} {speed if speed else 0:6.1f}")
print("\nđã chép vào:", destination)

## Nếu vướng

**Tràn bộ nhớ:** thêm `extra=("--load-4bit",)`. Còn tràn nữa thì thêm
`extra=("--load-4bit", "--gpu-memory", "12GiB")` — nhưng đẩy sang RAM làm chậm rất nhiều,
nên chỉ dùng khi cùng đường.

**Lỗi 401 hoặc 403 khi tải model:** token chưa đồng ý giấy phép Gemma. Vào trang model trên
Hugging Face bấm đồng ý, rồi chạy lại ô đăng nhập.

**Phiên bị ngắt giữa chừng:** kết quả của những model đã chấm xong vẫn nằm trong
`/kaggle/working/ket-qua-llm` nếu bạn đã chạy ô gom kết quả. Nên chạy ô đó sau mỗi model
thay vì để tới cuối.